# Avance 3 · Permeabilidad de la barrera con huellas

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [03](03_Baseline.ipynb) y el equivalente, para la fase 1, del 04c de la fase 2. Repite los seis modelos individuales del Avance 4 (regresión logística, árbol, vecinos, Bayes ingenuo, SVM con núcleo RBF y red de una capa) con **las mismas filas, los mismos pliegues y las mismas semillas**, y cambia una sola cosa: qué información de la estructura reciben.

Los modelos de las dos fases se entrenaron solo con descriptores de RDKit, que resumen la molécula entera (su tamaño, su polaridad, cuántos donadores de hidrógeno tiene). La decisión [1.3](../docs/decisiones.md#d1-3) contemplaba también las huellas **Sort & Slice**, que dicen qué fragmentos concretos tiene, y son la base para explicar una predicción a nivel de subestructura. Que no entraran fue el valor por omisión del cargador, no una decisión. Aquí medimos qué cambia al agregarlas.

Es un cuaderno de apoyo, no un entregable. Usa los seis modelos del 04 y no los tres del 03 porque, si las huellas sirven, el Avance 5 las va a necesitar en los modelos con los que se compara; los ensambles (el bosque y el boosting del 03) quedan para el Avance 5.

## 1. Preparación

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"
DATOS = RAIZ / "notebooks" / "datos"

from src.evaluation.cv import evaluar
from src.features.huellas import AgregarHuellas, agrupar_iguales, identificadores_morgan
from src.features.seleccion import DepuracionDescriptores, SeleccionL1
from src.models import cargar, seleccion

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  macOS-27.0.1-arm64-arm-64bit
python   3.12.12
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### Las huellas y el código que las arma

Una huella **Sort & Slice** es una tabla de 0 y 1: una columna por subestructura (el entorno de radio 2 alrededor de un átomo, el mismo que ECFP4) y un 1 si la molécula la tiene. De todas las subestructuras que aparecen se conservan las **2,048 más frecuentes**. A diferencia de una huella plegada por hash, cada columna es **una** subestructura, y por eso una columna con peso alto se puede traducir a un fragmento concreto.

**El vocabulario se aprende en cada pliegue, con sus filas de entrenamiento.** Es lo que más importa de este cuaderno. Qué subestructuras son «las más frecuentes» depende de con qué filas se cuenten: si se cuentan con todas, la validación ya influyó en qué columnas existen, y eso es una fuga que no aparece en ninguna métrica. Por eso las moléculas se leen una sola vez y la elección de las 2,048 ocurre dentro de la canalización, en cada ajuste. La matriz de partida, con una columna por cada subestructura que aparece en alguna molécula, no se filtra por frecuencia antes de partir.

**Lo que se ejecuta es el código de `src/`**, que importa la celda de arriba. Se muestra aquí para que se lea sin abrir otro archivo, y la prueba `tests/test_03c_codigo.py` comprueba que lo mostrado y lo importado sean idénticos.

```python
def identificadores_morgan(smiles, radio: int = 2) -> tuple[sparse.csr_matrix, np.ndarray]:
    """Matriz binaria dispersa (filas × subestructuras distintas) y los identificadores.

    Una columna por subestructura, sin plegar, ordenadas por identificador. Cada SMILES se lee
    una vez aunque aparezca en varias filas. Falla si alguno no se puede leer: quitarlo en
    silencio desalinearía la matriz con la de descriptores.
    """
    smiles = np.asarray(smiles, dtype=object)
    unicos, inverso = np.unique(smiles, return_inverse=True)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radio)
    ids_por_molecula = []
    ilegibles = []
    for s in unicos:
        mol = a_mol(s)
        if mol is None:
            ilegibles.append(s)
            ids_por_molecula.append(np.empty(0, dtype=np.int64))
        else:
            ids_por_molecula.append(
                np.fromiter(gen.GetSparseCountFingerprint(mol).GetNonzeroElements().keys(),
                            dtype=np.int64))
    if ilegibles:
        raise ValueError(f"{len(ilegibles)} SMILES no se pueden leer (el primero: {ilegibles[0]!r})")
    identificadores = np.unique(np.concatenate(ids_por_molecula))
    columnas = np.concatenate([np.searchsorted(identificadores, ids) for ids in ids_por_molecula])
    filas = np.repeat(np.arange(len(unicos)), [len(ids) for ids in ids_por_molecula])
    por_molecula = sparse.csr_matrix(
        (np.ones(len(columnas), dtype=np.uint8), (filas, columnas)),
        shape=(len(unicos), len(identificadores)))
    return por_molecula[inverso], identificadores

def agrupar_iguales(huellas: sparse.csr_matrix, identificadores: np.ndarray
                    ) -> tuple[sparse.csr_matrix, np.ndarray, list[tuple[int, ...]]]:
    """Deja una sola columna por cada grupo de subestructuras que valen lo mismo en todas las filas.

    Dos subestructuras que siempre aparecen juntas (el `S` y el `O=S` de un sulfonilo, por ejemplo)
    son la misma variable para cualquier modelo, y si ambas entran el modelo reparte su peso entre
    ellas de forma arbitraria. No se mira la etiqueta, y lo que es idéntico en todas las filas lo es
    también en cualquier subconjunto, así que hacerlo una vez sobre todo el conjunto no filtra
    información de la validación. Se queda la primera columna de cada grupo y se devuelve, para cada
    columna conservada, la tupla con los identificadores de todas las que representa (la propia
    primero), para poder nombrar la variable con todas sus subestructuras.
    """
    columnas = huellas.tocsc()
    columnas.sort_indices()
    grupos: dict[bytes, list[int]] = {}
    for j in range(columnas.shape[1]):
        grupos.setdefault(columnas.indices[columnas.indptr[j]:columnas.indptr[j + 1]].tobytes(),
                          []).append(j)
    conservadas = [g[0] for g in grupos.values()]
    equivalentes = [tuple(int(i) for i in identificadores[g]) for g in grupos.values()]
    return columnas[:, conservadas].tocsr(), identificadores[conservadas], equivalentes

class AgregarHuellas(BaseEstimator, TransformerMixin):
    """Entrada mixta → matriz densa con las columnas `densas` y las `n_bits` huellas más frecuentes.

    `X` es una matriz dispersa cuyas primeras columnas son las densas (descriptores y diana) y el
    resto es la salida de `identificadores_morgan`. En `fit` cuenta cuántas filas de entrenamiento
    tiene cada subestructura y se queda con las `n_bits` más frecuentes (los empates se resuelven por
    la posición de la columna, que sigue el orden del identificador, para que sea reproducible).
    `transform` devuelve las columnas densas indicadas y esas huellas, en `float32`.

    Con `densas=()` solo salen las huellas.
    """

    def __init__(self, n_densas: int, densas: tuple[int, ...] | None = None, n_bits: int = 2048):
        self.n_densas = n_densas
        self.densas = densas
        self.n_bits = n_bits

    def fit(self, X, y=None):
        huellas = sparse.csr_matrix(X)[:, self.n_densas:]
        frecuencia = np.asarray(huellas.sum(axis=0)).ravel()
        orden = np.lexsort((np.arange(len(frecuencia)), -frecuencia.astype(np.int64)))
        usadas = orden[:self.n_bits]
        self.columnas_ = usadas[frecuencia[usadas] > 0]
        return self

    def transform(self, X):
        X = sparse.csr_matrix(X)
        densas = np.arange(self.n_densas) if self.densas is None else np.asarray(self.densas, int)
        partes = [X[:, densas].toarray().astype(np.float32),
                  X[:, self.n_densas:][:, self.columnas_].toarray().astype(np.float32)]
        return np.hstack(partes)
```

La depuración de descriptores (brazo `depurados`) usa los dos filtros que el Avance 2 aplicó una sola vez, y la selección por coeficientes (brazo `seleccionados`) usa un Lasso. Aquí se ajustan **dentro de cada pliegue**:

```python
def poder_spearman(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """|Spearman| de cada columna contra un objetivo continuo: la versión de `poder_univariado`
    para una regresión. Una columna constante da 0."""
    rx = np.apply_along_axis(rankdata, 0, X)
    ry = rankdata(y)
    rx = rx - rx.mean(axis=0)
    ry = ry - ry.mean()
    den = np.sqrt((rx ** 2).sum(axis=0) * (ry ** 2).sum())
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (rx * ry[:, None]).sum(axis=0) / den
    return np.abs(np.nan_to_num(r))

class DepuracionDescriptores(BaseEstimator, TransformerMixin):
    """Los dos filtros de siempre, pero aprendidos con las filas de entrenamiento de cada pliegue.

    Actúa sobre las primeras `n_descriptores` columnas y deja pasar el resto (la diana en 0/1).
    Quita las casi constantes (un valor reúne `frecuencia_modal` de las filas) y, de cada par con
    |Spearman| mayor que `umbral`, la que menos se relaciona con el objetivo. Las `protegidas`
    (posiciones dentro de los descriptores) ganan cualquier par. Es el criterio de
    `casi_constantes` y `filtro_correlacion`; lo que cambia es dónde se ajusta.
    """

    def __init__(self, n_descriptores: int, protegidas: tuple[int, ...] = (),
                 frecuencia_modal: float = 0.99, umbral: float = 0.95):
        self.n_descriptores = n_descriptores
        self.protegidas = protegidas
        self.frecuencia_modal = frecuencia_modal
        self.umbral = umbral

    def fit(self, X, y):
        desc = np.asarray(X[:, :self.n_descriptores], dtype=float)
        variables = np.flatnonzero(casi_constantes(desc, self.frecuencia_modal))
        self.quitadas_constantes_ = np.setdiff1d(np.arange(self.n_descriptores), variables)
        sub = desc[:, variables]
        en_sub = tuple(int(np.flatnonzero(variables == p)[0]) for p in self.protegidas
                       if p in variables)
        sin_redundancia = filtro_correlacion(sub, poder_spearman(sub, np.asarray(y, float)),
                                             self.umbral, en_sub)
        self.descriptores_ = variables[sin_redundancia]
        self.quitadas_redundantes_ = variables[~sin_redundancia]
        self.columnas_ = np.r_[self.descriptores_, np.arange(self.n_descriptores, X.shape[1])]
        return self

    def transform(self, X):
        return X[:, self.columnas_]

class SeleccionL1(BaseEstimator, TransformerMixin):
    """Se queda con los descriptores a los que una penalización L1 no les lleva el peso a cero.

    Actúa sobre las primeras `n_descriptores` columnas y deja pasar el resto (la diana en 0/1).
    En cada ajuste escala los descriptores, entrena un Lasso contra el objetivo con las filas de
    entrenamiento y conserva los de coeficiente distinto de cero. Las `protegidas` se quedan
    siempre. `alpha` es un hiperparámetro que se fija de antemano; no se elige mirando la
    validación.
    """

    def __init__(self, n_descriptores: int, alpha: float = 0.01, protegidas: tuple[int, ...] = ()):
        self.n_descriptores = n_descriptores
        self.alpha = alpha
        self.protegidas = protegidas

    def fit(self, X, y):
        desc = np.asarray(X[:, :self.n_descriptores], dtype=float)
        lasso = Lasso(alpha=self.alpha, max_iter=5000).fit(
            StandardScaler().fit_transform(desc), np.asarray(y, float))
        elegidas = set(np.flatnonzero(lasso.coef_ != 0)) | set(self.protegidas)
        self.descriptores_ = np.array(sorted(elegidas), dtype=int)
        self.quitadas_ = np.setdiff1d(np.arange(self.n_descriptores), self.descriptores_)
        self.columnas_ = np.r_[self.descriptores_, np.arange(self.n_descriptores, X.shape[1])]
        return self

    def transform(self, X):
        return X[:, self.columnas_]
```

### Los datos

Son los mismos de la §2 del 03 y del 04: entrenamiento y validación (la prueba reservada no se toca), agrupados por andamio y esqueleto de conectividad, y desglosados por **origen** —natural exacto, análogo estereoquímico o sin equivalente natural—, que es el desglose de la decisión [1.6](../docs/decisiones.md#d1-6). El brazo de control usa la misma matriz de descriptores que el 03 y el 04, y sus seis modelos, la misma corrida que registró Antonio el 24 de octubre en `experimentos/2026-10-24-antonio-1.5-comparativa-6-modelos-fase-1/`.

Los brazos `depurados` y `seleccionados` parten de los descriptores **antes** del filtro del Avance 2 (`bbb_descriptores.parquet`), porque el filtro de ese avance se ajustó una vez con todo el entrenamiento y aquí hay que repetirlo dentro de cada pliegue. Esa matriz no incluye las tres columnas derivadas que el 02 agrega antes de filtrar.

In [2]:
t = cargar("permeabilidad", estratos="origen")
y, g, origen = t.y, t.grupos, t.estratos
X_control = t.X                                  # la matriz del 03
smiles = t.meta["SMILES"].to_numpy()

meta_completa = pd.read_parquet(PROCESADOS / "bbb_meta.parquet")
filas = np.flatnonzero(meta_completa["particion"].ne("prueba").to_numpy())
crudos = pd.read_parquet(PROCESADOS / "bbb_descriptores.parquet")
cols_crudos = list(crudos.columns)
X_crudos = crudos.to_numpy(np.float32)[filas]    # los descriptores antes del filtro del Avance 2
assert len(smiles) == len(y) == X_crudos.shape[0] == X_control.shape[0]

N_CONTROL = X_control.shape[1]
N_CRUDOS = X_crudos.shape[1]
print(f"{len(y):,} compuestos · {y.mean():.1%} BBB+ · {len(np.unique(g)):,} grupos · "
      f"{N_CONTROL} descriptores en el 03, {N_CRUDOS} antes de su filtro")
print(pd.Series(origen).value_counts().to_string())

6,306 compuestos · 65.0% BBB+ · 1,048 grupos · 151 descriptores en el 03, 203 antes de su filtro
natural exacto             2920
sin equivalente natural    2618
análogo estereoquímico      768


### Las huellas de los compuestos

Se leen los SMILES una sola vez y se arma una matriz dispersa con una columna por subestructura. **Todavía no se elige ningún vocabulario**: eso pasa dentro de cada pliegue. Antes se agrupan las subestructuras que valen lo mismo en todos los compuestos (porque siempre aparecen juntas, como el `S` y el `O=S` de un sulfonilo): se queda una sola columna y se guarda, en `equivalentes`, el nombre de todas las que representa. Eso no mira la etiqueta, y lo que es idéntico en todas las filas lo es en cualquier pliegue, así que no hay fuga. Así las 2,048 columnas no se gastan en la misma variable repetida y una atribución no reparte su peso entre copias.

In [3]:
t0 = time.perf_counter()
huellas, identificadores = identificadores_morgan(smiles)
n_distintas = huellas.shape[1]
huellas, identificadores, equivalentes = agrupar_iguales(huellas, identificadores)
n_agrupadas = sum(len(e) > 1 for e in equivalentes)
X_mixta = sparse.hstack([sparse.csr_matrix(X_control), huellas]).tocsr()
print(f"{n_distintas:,} subestructuras distintas en {huellas.shape[0]:,} compuestos; "
      f"{n_distintas - huellas.shape[1]:,} valían lo mismo que otra y quedan {huellas.shape[1]:,} columnas "
      f"({n_agrupadas} de ellas representan a más de una) · {time.perf_counter() - t0:.1f} s")

13,817 subestructuras distintas en 6,306 compuestos; 7,612 valían lo mismo que otra y quedan 6,205 columnas (2287 de ellas representan a más de una) · 0.4 s


## 2. Cinco brazos con los mismos seis modelos

| Brazo | Qué recibe el modelo | Para qué está |
|:--|:--|:--|
| `descriptores` | Los descriptores del 03 | Es el 04 en la fase 1. Es el **control**: tiene que repetir las cifras de la corrida registrada |
| `depurados` | Los descriptores previos al filtro del Avance 2, sin las columnas casi constantes ni las redundantes (\|Spearman\| mayor que 0.95), aprendido en cada pliegue | Responde si el filtro de una sola vez dejaba fuga o dejaba de más. TPSA y los donadores de hidrógeno se protegen, porque son las dos propiedades con las que se comprueba que una explicación tenga sentido |
| `seleccionados` | Esos mismos descriptores, de los que un Lasso (penalización L1, `alpha = 0.003`, sobre la etiqueta 0/1) conserva los que no lleva a cero | La otra forma de preguntar si sobran columnas: que sea un modelo el que elija. Se ajusta en cada pliegue. `alpha` se fijó de antemano para conservar cerca de la mitad (unos 90 de los 203 en un pliegue de entrenamiento, sin mirar ninguna validación), que es la fracción que conserva el de la fase 2 |
| `descriptores+huellas` | Los descriptores del 03 y las 2,048 huellas | Lo que se quiere medir: qué agregan los fragmentos a lo que ya había |
| `huellas` | Solo las 2,048 huellas | Cuánto dice la estructura por sí sola, sin los descriptores |

Los modelos son los de `src/models/alternativos.py`, sin afinar y sin cambios, los mismos seis del 04 (el SVM con núcleo RBF va sin afinar, como eligió el 04). Los que dependen de distancias o de magnitudes llevan el escalado mínimo-máximo dentro de su canalización, así que cada pliegue aprende el suyo. En las columnas de huellas, que ya valen 0 o 1, ese escalado no cambia nada.

**Lo que se fija antes de ver los resultados.**

- **Las métricas principales son el MCC y el AUPRC de la clase minoritaria**, como en el resto de la fase 1 (decisión C.1); el AUROC se reporta como tercera lectura y no decide. Se compara **pliegue por pliegue contra el brazo `descriptores`**: las diferencias se calculan sobre las mismas 15 evaluaciones (3 semillas × 5 pliegues), no entre promedios.
- **Un brazo mejora a un modelo solo si** la diferencia media supera el umbral **y** es positiva en al menos **12 de las 15** evaluaciones, **en las dos métricas principales a la vez**. El umbral de cada métrica es la mayor desviación de esa métrica entre las 15 evaluaciones del brazo de control, entre los seis modelos, y se calcula con ese brazo solo, antes de mirar los demás. *Es una regla provisional que el equipo confirma o cambia antes de leer los resultados, y se fijó después de ver cómo se comporta el control en un pliegue, así que no está pre-registrada.* Si no se cumple, se reporta como «sin mejora medible». Si las huellas no mejoran la métrica, el argumento a favor de usarlas sería la explicabilidad, y así se diría; no se presentaría como una mejora.
- **Los 15 pares no son independientes**: las semillas comparten filas y los pliegues parten los mismos datos. Los 12 de 15 son una prueba de signos con esa limitación, no una probabilidad.
- **Los desgloses por origen son exploratorios.** La comparación que decide es la del agregado. Los estratos se reportan porque las huellas podrían ayudar o delatar de qué familia viene un compuesto, y se leen como tales.
- **Contra el sobreajuste**, además de los pliegues agrupados por esqueleto, se mide la **brecha entre entrenamiento y validación** de cada modelo con cada brazo, en AUROC porque es el diagnóstico y no la decisión.

In [4]:
SEMILLAS = (0, 1, 2)
N_BITS = 2048                                    # tamaño del vocabulario de Sort & Slice
ALFA_L1 = 0.003                                  # penalización del Lasso; fija, no se elige con la validación
PROTEGIDAS = tuple(cols_crudos.index(c) for c in ("TPSA", "NumHDonors"))   # posiciones dentro de los descriptores
MINIMO_PLIEGUES = 12                             # de 15 evaluaciones, en cuántas tiene que mejorar
MUESTRA_ENTRENAMIENTO = 3000                     # filas de entrenamiento para medir la brecha
CLAVES = ("logistica", "arbol", "knn", "bayes", "svm_rbf", "mlp")   # los seis del 04, sin afinar
ORDEN_MODELOS = list(CLAVES)
PRINCIPALES = ("mcc", "auprc_minoritaria")       # las que deciden; el AUROC no

# Cada brazo: la matriz que recibe y cómo se arma su canalización a partir del modelo del 04.
BRAZOS = {
    "descriptores":         (X_control, lambda m: m),
    "depurados":            (X_crudos,  lambda m: make_pipeline(DepuracionDescriptores(N_CRUDOS, PROTEGIDAS), m)),
    "seleccionados":        (X_crudos,  lambda m: make_pipeline(SeleccionL1(N_CRUDOS, ALFA_L1, PROTEGIDAS), m)),
    "descriptores+huellas": (X_mixta,   lambda m: make_pipeline(AgregarHuellas(N_CONTROL, n_bits=N_BITS), m)),
    "huellas":              (X_mixta,   lambda m: make_pipeline(AgregarHuellas(N_CONTROL, (), N_BITS), m)),
}
orden_brazos = list(BRAZOS)

In [5]:
partes = []
inicio = time.perf_counter()
for brazo, (matriz, armar) in BRAZOS.items():
    pipes = {nombre: armar(p) for nombre, p in seleccion(CLAVES).items()}
    r, _ = evaluar(pipes, matriz, y, g, estratos=origen, semillas=SEMILLAS, verboso=False)
    partes.append(r.assign(brazo=brazo))
    print(f"{brazo:<22} {(time.perf_counter() - inicio) / 60:6.1f} min", flush=True)
res = pd.concat(partes, ignore_index=True)

descriptores              0.9 min


depurados                 1.9 min


seleccionados             2.8 min


descriptores+huellas      8.7 min


huellas                  14.5 min


## 3. Resultados

### Cómo se fueron quitando variables

Esta tabla cuenta cada paso por el que pasa una columna, desde las que trae el archivo hasta las que ve un modelo, y quién la quitó:

- **Filtro del Avance 2.** Se hizo una sola vez con todo el entrenamiento, antes de los pliegues. Es el que produce la matriz del 03.
- **`depurados`.** Se ajusta en cada pliegue, en dos pasos: primero las casi constantes (un solo valor reúne el 99% de las filas) y luego las redundantes (de cada par con \|Spearman\| mayor que 0.95 se queda la que más se relaciona con la etiqueta).
- **`seleccionados`.** Se ajusta en cada pliegue: se queda con lo que un Lasso no manda a cero.
- **Huellas.** Aquí no se quita ninguna por criterio estadístico: se agrupan las idénticas una vez y en cada pliegue se conservan las 2,048 más frecuentes.

Como cada pliegue elige con sus propias filas, las cuentas son un promedio con el mínimo y el máximo entre las 15 evaluaciones.

In [6]:
quitadas, elegidas = {}, {}
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)         # los mismos pliegues de la evaluación
    for k, (ajuste, _) in enumerate(cv.split(X_crudos, y, g), 1):
        dep = DepuracionDescriptores(N_CRUDOS, PROTEGIDAS).fit(X_crudos[ajuste], y[ajuste])
        l1 = SeleccionL1(N_CRUDOS, ALFA_L1, PROTEGIDAS).fit(X_crudos[ajuste], y[ajuste])
        quitadas[(semilla, k)] = {"casi constantes": len(dep.quitadas_constantes_),
                                  "redundantes": len(dep.quitadas_redundantes_),
                                  "Lasso en cero": len(l1.quitadas_)}
        elegidas[("depurados", semilla, k)] = [cols_crudos[i] for i in dep.descriptores_]
        elegidas[("seleccionados", semilla, k)] = [cols_crudos[i] for i in l1.descriptores_]


def cuantas(paso):
    v = [q[paso] for q in quitadas.values()]
    return f"{np.mean(v):.0f} ({min(v)}–{max(v)})", np.mean(v)


q_const, m_const = cuantas("casi constantes")
q_red, m_red = cuantas("redundantes")
q_l1, m_l1 = cuantas("Lasso en cero")
embudo = pd.DataFrame([
    ("Archivo de descriptores", "", "", N_CRUDOS),
    ("Filtro del Avance 2, una vez", "casi constantes y correlación de Spearman", f"{N_CRUDOS - N_CONTROL}", N_CONTROL),
    ("`depurados`: casi constantes", "un valor reúne el 99% de las filas", q_const, f"{N_CRUDOS - m_const:.0f}"),
    ("`depurados`: redundantes", "|Spearman| > 0.95 con otra", q_red, f"{N_CRUDOS - m_const - m_red:.0f}"),
    ("`seleccionados`: Lasso", f"coeficiente cero con alpha = {ALFA_L1}", q_l1, f"{N_CRUDOS - m_l1:.0f}"),
    ("Huellas: subestructuras distintas", "una columna por cada una", "", f"{n_distintas:,}"),
    ("Huellas: idénticas agrupadas", "valen lo mismo en todos los compuestos",
     f"{n_distintas - huellas.shape[1]:,}", f"{huellas.shape[1]:,}"),
    ("Huellas: las más frecuentes", "las 2,048 con más compuestos, en cada pliegue", "", f"{N_BITS:,}"),
], columns=["paso", "criterio", "quita (media y rango)", "quedan"]).set_index("paso")
embudo

,criterio,quita (media y rango),quedan
paso,,,
Archivo de descriptores,,,203
"Filtro del Avance 2, una vez",casi constantes y correlación de Spearman,52,151
`depurados`: casi constantes,un valor reúne el 99% de las filas,23 (21–25),180
`depurados`: redundantes,|Spearman| > 0.95 con otra,30 (28–31),150
`seleccionados`: Lasso,coeficiente cero con alpha = 0.003,113 (104–122),90
Huellas: subestructuras distintas,una columna por cada una,,"13,817"
Huellas: idénticas agrupadas,valen lo mismo en todos los compuestos,"7,612","6,205"
Huellas: las más frecuentes,"las 2,048 con más compuestos, en cada pliegue",,"2,048"


Y los descriptores que sobreviven en todas las evaluaciones, que son los que más se pueden defender como parte de la explicación:

In [7]:
resumen = []
for brazo in ("depurados", "seleccionados"):
    listas = [v for (b, _, _), v in elegidas.items() if b == brazo]
    veces = pd.Series([c for v in listas for c in v]).value_counts()
    resumen.append({"brazo": brazo, "evaluaciones": len(listas),
                    "descriptores_que_deja": f"{min(map(len, listas))}–{max(map(len, listas))} de {N_CRUDOS}",
                    "en_todas": int((veces == len(listas)).sum())})
display(pd.DataFrame(resumen).set_index("brazo"))
n_l1 = sum(1 for (b, _, _) in elegidas if b == "seleccionados")
cuenta_l1 = pd.Series([c for (b, _, _), v in elegidas.items() if b == "seleccionados" for c in v]).value_counts()
en_todas_l1 = list(cuenta_l1[cuenta_l1 == n_l1].index)
print(f"Descriptores que el Lasso conserva en todas las evaluaciones ({len(en_todas_l1)}):")
print(", ".join(en_todas_l1))

,evaluaciones,descriptores_que_deja,en_todas
brazo,,,
depurados,15,148–153 de 203,130
seleccionados,15,81–99 de 203,37


Descriptores que el Lasso conserva en todas las evaluaciones (37):
qed, FpDensityMorgan1, AvgIpc, PEOE_VSA11, PEOE_VSA13, PEOE_VSA5, PEOE_VSA6, PEOE_VSA9, SlogP_VSA10, SlogP_VSA7, TPSA, EState_VSA11, EState_VSA3, EState_VSA6, NumHDonors, NumRotatableBonds, NumSaturatedHeterocycles, NumSpiroAtoms, NumUnspecifiedAtomStereoCenters, RingCount, fr_Ar_COO, fr_Ar_OH, fr_COO, fr_HOCCN, fr_N_O, fr_SH, fr_aldehyde, fr_alkyl_halide, fr_allylic_oxid, fr_benzodiazepine, fr_hdrzone, fr_imide, fr_lactam, fr_morpholine, fr_nitroso, fr_piperdine, fr_quatN


### El control: el brazo `descriptores` repite el 04

Antes de leer nada, el brazo que no cambia nada tiene que dar lo que ya se registró en la corrida de los seis modelos de la fase 1. El registro guarda promedios redondeados a tres decimales, así que la diferencia esperada es de ese tamaño; si es mayor, algo en la preparación o en el entorno es distinto (otra versión de Python o de RDKit, por ejemplo) y los demás brazos no se pueden comparar con el 04.

In [8]:
REGISTRO = RAIZ / "experimentos" / "2026-10-24-antonio-1.5-comparativa-6-modelos-fase-1" / "metricas.json"
registrado = pd.DataFrame(json.loads(REGISTRO.read_text())["seis"]).T[["mcc", "auprc_minoritaria", "auroc"]]
aqui = (res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")]
        .groupby("modelo")[["mcc", "auprc_minoritaria", "auroc"]].mean())
control = pd.concat({"04 (registrado)": registrado, "03c (esta corrida)": aqui}, axis=1).loc[ORDEN_MODELOS]
for m in ("mcc", "auprc_minoritaria", "auroc"):
    control[("diferencia", m)] = control[("03c (esta corrida)", m)] - control[("04 (registrado)", m)]
control.round(3)

04 (registrado)                          03c (esta corrida)  \
                      mcc auprc_minoritaria  auroc                mcc   
logistica           0.688             0.872  0.913              0.687   
arbol               0.568             0.706  0.836              0.570   
knn                 0.669             0.863  0.914              0.669   
bayes               0.566             0.801  0.857              0.566   
svm_rbf             0.703             0.884  0.919              0.702   
mlp                 0.674             0.865  0.913              0.674   

                                   diferencia                           
          auprc_minoritaria  auroc        mcc auprc_minoritaria  auroc  
logistica             0.872  0.913     -0.001             0.000  0.000  
arbol                 0.700  0.831      0.002            -0.006 -0.005  
knn                   0.863  0.914      0.000             0.000  0.000  
bayes                 0.801  0.857     -0.000            -0.000  0.000  
svm_rbf               0.884  0.919     -0.001             0.000  0.000  
mlp                   0.864  0.911      0.000            -0.001 -0.002

### Todos los brazos, por modelo

El MCC, el AUPRC de la minoritaria y el AUROC medios de cada modelo en cada brazo, y después cuánto tarda un ajuste. La desviación es dispersión entre evaluaciones, no un error estándar: las semillas comparten filas.

In [9]:
def tabla(r, estrato="todos"):
    """Media por brazo y modelo de cada métrica, con la dispersión del MCC entre evaluaciones."""
    r = r[r["estrato"] == estrato]
    t = r.groupby(["brazo", "modelo"])[["mcc", "auprc_minoritaria", "auroc", "segundos_ajuste"]].mean()
    t.insert(1, "desv_mcc", r.groupby(["brazo", "modelo"])["mcc"].std())
    return t.round(4)


tabla_todos = tabla(res)
for m in ("mcc", "auprc_minoritaria", "auroc"):
    print(m)
    display(tabla_todos[m].unstack("brazo")[orden_brazos].loc[ORDEN_MODELOS])

mcc


brazo,descriptores,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,,
logistica,0.6866,0.6850,0.6832,0.6629,0.6237
arbol,0.5704,0.5669,0.5813,0.5848,0.5837
knn,0.6692,0.6794,0.6768,0.6505,0.6327
bayes,0.5660,0.5518,0.4021,0.5177,0.5103
svm_rbf,0.7017,0.7014,0.6976,0.6940,0.6629
mlp,0.6743,0.6739,0.6842,0.6441,0.6157


auprc_minoritaria


brazo,descriptores,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,,
logistica,0.8724,0.8714,0.8743,0.8534,0.8458
arbol,0.7001,0.7038,0.7137,0.7083,0.7346
knn,0.8633,0.8608,0.8598,0.8569,0.8416
bayes,0.8005,0.8009,0.7989,0.5639,0.5584
svm_rbf,0.8842,0.8860,0.8843,0.8742,0.8686
mlp,0.8640,0.8629,0.8603,0.8281,0.8168


auroc


brazo,descriptores,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,,
logistica,0.9133,0.9121,0.9132,0.9091,0.8960
arbol,0.8314,0.8336,0.8333,0.8356,0.8351
knn,0.9141,0.9096,0.9112,0.9085,0.8950
bayes,0.8573,0.8556,0.8715,0.7708,0.7669
svm_rbf,0.9190,0.9195,0.9194,0.9174,0.9063
mlp,0.9110,0.9110,0.9080,0.8964,0.8819


In [10]:
tabla_todos["segundos_ajuste"].unstack("brazo")[orden_brazos].loc[ORDEN_MODELOS].round(1)

brazo,descriptores,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,,
logistica,0.0,0.1,0.0,0.1,0.1
arbol,0.2,0.2,0.1,0.5,0.5
knn,0.0,0.1,0.0,0.0,0.0
bayes,0.0,0.1,0.0,0.0,0.0
svm_rbf,1.4,1.4,1.0,19.7,19.6
mlp,1.9,2.0,2.2,1.6,1.5


### ¿Mejoran las huellas? La comparación pliegue por pliegue

Para cada modelo y cada brazo se resta cada métrica del brazo `descriptores` en **la misma evaluación** (misma semilla, mismo pliegue). `mejor_en` cuenta en cuántas de las 15 evaluaciones el brazo gana, y `mejora` aplica la regla fijada arriba: las dos métricas principales cumplen el umbral y el mínimo. El umbral se calcula primero, solo con el brazo de control.

In [11]:
control_todos = res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")]
UMBRALES = {m: round(float(control_todos.groupby("modelo")[m].std().max()), 3) for m in PRINCIPALES}
print("Umbrales: la mayor desviación de cada métrica del control entre sus 15 evaluaciones")
print(UMBRALES)


def comparacion(r, estrato="todos"):
    """Diferencia de cada brazo contra `descriptores`, evaluación por evaluación, en cada métrica."""
    r = r[r["estrato"] == estrato]
    filas = []
    for metrica in ("mcc", "auprc_minoritaria", "auroc"):
        p = r.pivot_table(index=["modelo", "semilla", "pliegue"], columns="brazo", values=metrica)
        for brazo in orden_brazos[1:]:
            dif = (p[brazo] - p["descriptores"]).dropna()
            for modelo, v in dif.groupby("modelo"):
                filas.append({"brazo": brazo, "modelo": modelo, "metrica": metrica, "evaluaciones": len(v),
                              "dif_media": v.mean(), "dif_desv": v.std(), "mejor_en": int((v > 0).sum())})
    c = pd.DataFrame(filas).set_index(["brazo", "modelo", "metrica"])
    c["cumple"] = c["dif_media"].gt(c.index.get_level_values("metrica").map(UMBRALES).astype(float)) &         c["mejor_en"].ge(MINIMO_PLIEGUES)
    mejora = c["cumple"].unstack("metrica")[list(PRINCIPALES)].all(axis=1).rename("mejora")
    return c.round(4), mejora


comparacion_todos, mejora_todos = comparacion(res)
display(comparacion_todos.unstack("metrica")[["dif_media", "mejor_en"]].round(3))
mejora_todos.unstack("brazo")[orden_brazos[1:]].loc[ORDEN_MODELOS]

Umbrales: la mayor desviación de cada métrica del control entre sus 15 evaluaciones
{'mcc': 0.094, 'auprc_minoritaria': 0.058}


dif_media                \
metrica                        auprc_minoritaria  auroc    mcc   
brazo                modelo                                      
depurados            arbol                 0.004  0.002 -0.004   
                     bayes                 0.000 -0.002 -0.014   
                     knn                  -0.002 -0.004  0.010   
                     logistica            -0.001 -0.001 -0.002   
                     mlp                  -0.001 -0.000 -0.000   
                     svm_rbf               0.002  0.000 -0.000   
descriptores+huellas arbol                 0.008  0.004  0.014   
                     bayes                -0.237 -0.086 -0.048   
                     knn                  -0.006 -0.006 -0.019   
                     logistica            -0.019 -0.004 -0.024   
                     mlp                  -0.036 -0.015 -0.030   
                     svm_rbf              -0.010 -0.002 -0.008   
huellas              arbol                 0.034  0.004  0.013   
                     bayes                -0.242 -0.090 -0.056   
                     knn                  -0.022 -0.019 -0.036   
                     logistica            -0.027 -0.017 -0.063   
                     mlp                  -0.047 -0.029 -0.059   
                     svm_rbf              -0.016 -0.013 -0.039   
seleccionados        arbol                 0.014  0.002  0.011   
                     bayes                -0.002  0.014 -0.164   
                     knn                  -0.004 -0.003  0.008   
                     logistica             0.002 -0.000 -0.004   
                     mlp                  -0.004 -0.003  0.010   
                     svm_rbf               0.000  0.000 -0.004   

                                        mejor_en            
metrica                        auprc_minoritaria auroc mcc  
brazo                modelo                                 
depurados            arbol                    11     9   7  
                     bayes                     9     8   5  
                     knn                       5     4  11  
                     logistica                 6     4   5  
                     mlp                       9     5   7  
                     svm_rbf                  13    10   9  
descriptores+huellas arbol                     9     9   7  
                     bayes                     0     1   4  
                     knn                       5     7   5  
                     logistica                 3     7   7  
                     mlp                       2     3   5  
                     svm_rbf                   4     7   7  
huellas              arbol                    13     9   8  
                     bayes                     0     1   3  
                     knn                       4     3   2  
                     logistica                 3     3   3  
                     mlp                       2     3   2  
                     svm_rbf                   2     3   3  
seleccionados        arbol                    10     9   6  
                     bayes                     8    11   0  
                     knn                       6     6   8  
                     logistica                 8     9   5  
                     mlp                       6     6   9  
                     svm_rbf                   6     7   5

brazo,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,
logistica,False,False,False,False
arbol,False,False,False,False
knn,False,False,False,False
bayes,False,False,False,False
svm_rbf,False,False,False,False
mlp,False,False,False,False


### Por origen

Un promedio puede esconder cómo le va a los productos naturales. Los fragmentos de los naturales podrían ayudar donde los descriptores no, o podrían delatar de qué familia viene el compuesto sin decir nada de si atraviesa la barrera. Por eso se mira cada origen por separado, **como lectura exploratoria**: la comparación que decide es la del agregado.

In [12]:
por_origen = {e: comparacion(res, e)[0] for e in pd.unique(res.loc[res["estrato"] != "todos", "estrato"])}
mcc_origen = pd.concat({e: tabla(res, e)["mcc"].unstack("brazo")[orden_brazos]
                        for e in por_origen}, names=["origen"])
mcc_origen

brazo                              descriptores  depurados  seleccionados  \
origen                  modelo                                              
sin equivalente natural arbol            0.3638     0.3596         0.3922   
                        bayes            0.3431     0.3528         0.2580   
                        knn              0.4762     0.4951         0.4978   
                        logistica        0.5001     0.5014         0.4932   
                        mlp              0.5406     0.4928         0.5209   
                        svm_rbf          0.4989     0.5139         0.5147   
natural exacto          arbol            0.5993     0.6002         0.5954   
                        bayes            0.6127     0.5887         0.4663   
                        knn              0.6935     0.7013         0.6987   
                        logistica        0.6924     0.6885         0.6931   
                        mlp              0.6703     0.6832         0.6932   
                        svm_rbf          0.7182     0.7135         0.7090   
análogo estereoquímico  arbol            0.5986     0.5989         0.6079   
                        bayes            0.6391     0.6279         0.4548   
                        knn              0.6187     0.6255         0.6451   
                        logistica        0.6916     0.6902         0.6692   
                        mlp              0.6558     0.6752         0.6779   
                        svm_rbf          0.6958     0.6941         0.6864   

brazo                              descriptores+huellas  huellas  
origen                  modelo                                    
sin equivalente natural arbol                    0.4166   0.3890  
                        bayes                    0.3939   0.3868  
                        knn                      0.5102   0.4919  
                        logistica                0.5083   0.4838  
                        mlp                      0.4941   0.4646  
                        svm_rbf                  0.5478   0.5113  
natural exacto          arbol                    0.6194   0.6119  
                        bayes                    0.5326   0.5235  
                        knn                      0.6606   0.6434  
                        logistica                0.6832   0.6345  
                        mlp                      0.6596   0.6253  
                        svm_rbf                  0.7070   0.6717  
análogo estereoquímico  arbol                    0.5458   0.5613  
                        bayes                    0.5037   0.5046  
                        knn                      0.5757   0.5539  
                        logistica                0.5991   0.5580  
                        mlp                      0.5834   0.5541  
                        svm_rbf                  0.6134   0.5853

In [13]:
pd.concat(por_origen, names=["origen"]).query("metrica in @PRINCIPALES")[["dif_media", "dif_desv", "mejor_en", "cumple"]]

dif_media  \
origen                  brazo     modelo    metrica                        
sin equivalente natural depurados arbol     mcc                  -0.0042   
                                  bayes     mcc                   0.0096   
                                  knn       mcc                   0.0190   
                                  logistica mcc                   0.0013   
                                  mlp       mcc                  -0.0478   
...                                                                  ...   
análogo estereoquímico  huellas   bayes     auprc_minoritaria    -0.1545   
                                  knn       auprc_minoritaria    -0.0680   
                                  logistica auprc_minoritaria    -0.0214   
                                  mlp       auprc_minoritaria    -0.0436   
                                  svm_rbf   auprc_minoritaria    -0.0102   

                                                               dif_desv  \
origen                  brazo     modelo    metrica                       
sin equivalente natural depurados arbol     mcc                  0.0666   
                                  bayes     mcc                  0.0732   
                                  knn       mcc                  0.0460   
                                  logistica mcc                  0.0164   
                                  mlp       mcc                  0.0738   
...                                                                 ...   
análogo estereoquímico  huellas   bayes     auprc_minoritaria    0.1000   
                                  knn       auprc_minoritaria    0.0711   
                                  logistica auprc_minoritaria    0.0560   
                                  mlp       auprc_minoritaria    0.0673   
                                  svm_rbf   auprc_minoritaria    0.0393   

                                                               mejor_en  \
origen                  brazo     modelo    metrica                       
sin equivalente natural depurados arbol     mcc                       7   
                                  bayes     mcc                       6   
                                  knn       mcc                       8   
                                  logistica mcc                       6   
                                  mlp       mcc                       6   
...                                                                 ...   
análogo estereoquímico  huellas   bayes     auprc_minoritaria         1   
                                  knn       auprc_minoritaria         3   
                                  logistica auprc_minoritaria         5   
                                  mlp       auprc_minoritaria         4   
                                  svm_rbf   auprc_minoritaria         5   

                                                               cumple  
origen                  brazo     modelo    metrica                    
sin equivalente natural depurados arbol     mcc                 False  
                                  bayes     mcc                 False  
                                  knn       mcc                 False  
                                  logistica mcc                 False  
                                  mlp       mcc                 False  
...                                                               ...  
análogo estereoquímico  huellas   bayes     auprc_minoritaria   False  
                                  knn       auprc_minoritaria   False  
                                  logistica auprc_minoritaria   False  
                                  mlp       auprc_minoritaria   False  
                                  svm_rbf   auprc_minoritaria   False  

[144 rows x 4 columns]

### La brecha entre entrenamiento y validación

Un modelo que se aprende los ejemplos acierta mucho en lo que vio y mucho menos en lo que no. La brecha es el AUROC sobre una muestra de 3,000 filas de entrenamiento menos el de validación, con la semilla 0 y sus cinco pliegues. Con más columnas la brecha crece si el modelo está sobreajustando; que crezca no es un defecto si la validación también mejora. Los dos modelos de árboles ya sobreajustaban en el 03, así que se compara cada modelo contra sí mismo en los demás brazos.

In [14]:
filas_brecha = []
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
for k, (ajuste, val) in enumerate(cv.split(X_control, y, g), 1):
    muestra = np.random.default_rng(k).choice(ajuste, min(MUESTRA_ENTRENAMIENTO, len(ajuste)), replace=False)
    for brazo, (matriz, armar) in BRAZOS.items():
        for nombre, p in seleccion(CLAVES).items():
            m = armar(p).fit(matriz[ajuste], y[ajuste])
            filas_brecha.append({"brazo": brazo, "modelo": nombre, "pliegue": k,
                                 "auroc_entrenamiento": roc_auc_score(y[muestra], m.predict_proba(matriz[muestra])[:, 1]),
                                 "auroc_validacion": roc_auc_score(y[val], m.predict_proba(matriz[val])[:, 1])})
brecha = pd.DataFrame(filas_brecha).groupby(["brazo", "modelo"])[["auroc_entrenamiento", "auroc_validacion"]].mean()
brecha["brecha"] = brecha["auroc_entrenamiento"] - brecha["auroc_validacion"]
brecha.round(4).unstack("brazo")["brecha"][orden_brazos].loc[ORDEN_MODELOS]

brazo,descriptores,depurados,seleccionados,descriptores+huellas,huellas
modelo,,,,,
logistica,0.0293,0.0306,0.0296,0.0803,0.0912
arbol,0.1612,0.1590,0.1569,0.1727,0.1520
knn,0.0525,0.0556,0.0567,0.0558,0.0688
bayes,0.0212,0.0235,0.0339,0.0796,0.0816
svm_rbf,0.0405,0.0389,0.0376,0.0680,0.0762
mlp,0.0806,0.0744,0.0849,0.1037,0.1192


In [15]:
# Registro en experimentos/ (decisión 1.3: qué representación reciben los modelos).
from src.tracking import Corrida

with Corrida("comparativa-permeabilidad-huellas", decision="1.3",
             params={"modelos": list(CLAVES), "brazos": list(BRAZOS), "pliegues": 5,
                     "metricas_principales": list(PRINCIPALES), "metrica_tercera_lectura": "auroc", "n_bits_sortslice": N_BITS, "radio_morgan": 2,
                     "vocabulario": "se aprende en cada ajuste",
                     "depuracion": {"frecuencia_modal": 0.99, "umbral_spearman": 0.95,
                                    "protegidas": ["TPSA", "NumHDonors"]},
                     "seleccion_l1": {"alpha": ALFA_L1, "protegidas": ["TPSA", "NumHDonors"]},
                     "regla_mejora": {"umbral_por_metrica": UMBRALES, "minimo_evaluaciones": MINIMO_PLIEGUES, "de": 15,
                                      "criterio": "las dos metricas principales a la vez",
                                      "umbral": "mayor desviacion del control entre sus 15 evaluaciones",
                                      "comparacion_decisiva": "agregado contra descriptores",
                                      "estratos": "exploratorios",
                                      "estado": "provisional; fijada despues de ver el control en un pliegue; no pre-registrada"},
                     "control_registrado": "2026-10-24-antonio-1.5-comparativa-6-modelos-fase-1"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "bbb_meta.parquet", PROCESADOS / "bbb_descriptores.parquet",
                    PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet"]) as c:
    c.metricas({"todos": {f"{b}/{m}": v for (b, m), v in tabla_todos.to_dict(orient="index").items()},
                "comparacion_todos": {f"{b}/{m}/{k}": v for (b, m, k), v in
                                      comparacion_todos.to_dict(orient="index").items()},
                "brecha": {f"{b}/{m}": v for (b, m), v in brecha.to_dict(orient="index").items()},
                **{e: {f"{b}/{m}": v for (b, m), v in tabla(res, e).to_dict(orient="index").items()}
                   for e in por_origen}})

print(c.carpeta.name)

2026-10-28-artemio-1.3-comparativa-permeabilidad-huellas


In [16]:
# Las 15 evaluaciones por brazo, modelo y estrato. El cuaderno lo **escribe** y no lo lee: las
# conclusiones salen de las tablas de arriba. El CSV es para rehacer la lectura 3.1 y las
# diferencias pareadas sin repetir la corrida.
# Va después del registro a propósito: escribirlo antes dejaría el árbol sucio al registrar.
SALIDA = RAIZ / "notebooks" / "datos" / "03c_brazos.csv"
res.to_csv(SALIDA, index=False)
print(f"{SALIDA.relative_to(RAIZ)} · {len(res):,} filas")

notebooks/datos/03c_brazos.csv · 1,800 filas


## 3.1 Lectura

**En la fase 1 las huellas no mejoran a ningún modelo, y la matriz del 03 no sobra.** Es lo
contrario de lo que midió el [04c](04c_Regresion_actividad_huellas.ipynb) en la fase 2, y las dos
cosas son compatibles: allá se predice la potencia de un par compuesto–diana y aquí una propiedad
de transporte que los descriptores físico-químicos ya describen bien.

### El control

El brazo `descriptores` reproduce la corrida registrada de los seis modelos en **17 de las 18
casillas**, con diferencias de 0.000 a 0.002. Queda una sola excepción:

| Casilla | Diferencia | De qué es |
|:--|--:|:--|
| `arbol`, AUPRC y AUROC | −0.006, −0.005 | el entorno registrado difiere en más que el sistema operativo (Linux con 3.12.15 contra macOS con 3.12.12): no coinciden el sha256 de `bbb_descriptores_filtrados_sin_normalizar.parquet` ni el del lock, así que la matriz de entrada no es la misma. Un árbol decide por umbrales, así que es el más sensible a una diferencia pequeña en los valores, y es el único de los seis que se mueve. Cuál de las dos matrices es la canónica está por decidir |

La casilla número 18 era el AUPRC de `bayes`, y hacía falta arreglar el arnés para que
reprodujera. `orientar_minoritaria` puntuaba la clase minoritaria con `1 - p`, y toda `p` menor
que la mitad del épsilon de la máquina da `1 - p == 1.0`: las filas que el modelo considera
minoritarias con más confianza quedaban empatadas en la cima del orden, y con ellas filas de la
mayoritaria con `p` igual de pequeña. En un pliegue son 504 filas empatadas, 154 de ellas de la
clase mayoritaria. GaussianNB es el único de los seis modelos que satura así. Negar es exacto y
restar no, así que la función devuelve `-p`; con eso `bayes` da 0.801, que es el valor registrado.
Esa corrección es parte de esta rama y nunca afectó al AUROC de la tabla, que se calcula con `p`
sin voltear.

### La regla no la pasa nadie

Los umbrales los fijó el control antes de mirar los demás brazos, y salieron altos porque el MCC
de la fase 1 varía mucho entre pliegues: **0.094 de MCC y 0.058 de AUPRC**. Con eso, **ningún
brazo mejora a ningún modelo**.

El caso que más se acerca es el árbol con `huellas` en AUPRC: **+0.035 de diferencia pareada,
mejor en 13 de las 15 evaluaciones**. Cumple el mínimo de pliegues y no el umbral, y además el
MCC del mismo brazo y modelo sube solo 0.013, así que tampoco cumpliría el requisito de las dos
métricas a la vez. La regla es conservadora a propósito y el umbral se calculó antes de ver esto;
no se baja.

En dirección contraria, casi todo el deterioro también cae dentro del umbral, así que lo que queda
establecido es «sin mejora medible» y no «empeoran». Las tres excepciones son de un solo modelo:

| Brazo | Modelo | Métrica | Diferencia | Peor en |
|:--|:--|:--|--:|--:|
| `huellas` | `bayes` | AUPRC | −0.242 | 15 de 15 |
| `descriptores+huellas` | `bayes` | AUPRC | −0.237 | 15 de 15 |
| `seleccionados` | `bayes` | MCC | −0.164 | 15 de 15 |

`bayes` asume que las variables son independientes dentro de cada clase, y 2,048 columnas de
fragmentos que coocurren son el caso donde ese supuesto falla más; que sea además el modelo cuyo
AUPRC es frágil obliga a leer sus dos filas de AUPRC con cuidado, pero la caída del MCC no depende
de eso.

### Las huellas no sustituyen a los descriptores

En la fase 2 el brazo `huellas` quedaba a 0.002 del brazo ancho: los fragmentos reemplazaban a los
descriptores. Aquí no. `descriptores+huellas` supera a `huellas` en los seis modelos, y por un
margen que no es ruido en cuatro de ellos (logística +0.039, svm_rbf +0.031, mlp +0.028,
knn +0.018 de MCC). La estructura por sí sola no alcanza para la permeabilidad, lo que es
coherente con que TPSA y los donadores de hidrógeno dominen empíricamente: son propiedades de la
molécula entera, no la presencia de un fragmento.

### Lo que sí cambia es el sobreajuste

Las huellas no mejoran la validación y sí ensanchan la brecha. Con `huellas` se triplica en la
logística (0.029 → 0.091) y en `bayes` (0.021 → 0.082), y casi se duplica en svm_rbf
(0.041 → 0.076) y en el mlp (0.081 → 0.119). El árbol es el único que no la ensancha
(0.161 → 0.152), y ya era el más sobreajustado de los seis en todos los brazos.

Quitar descriptores no cambia nada. `depurados` deja 148–153 de 203 y `seleccionados` 81–99
(el Lasso con `alpha = 0.003` conserva 90 en promedio, cerca de la mitad como se buscaba, y 37
sobreviven en las 15 evaluaciones). Ninguno de los dos mejora ni empeora a ningún modelo más allá
del umbral, salvo `bayes` en `seleccionados`. Dicho de otro modo: de los 151 descriptores del 03
se puede prescindir de un tercio sin perder nada medible, pero no se gana nada por hacerlo.

### Tiempo

Barato, salvo uno. El SVM con kernel pasa de 1.4 s por ajuste con descriptores a **20 s** con
huellas, y se lleva las tres cuartas partes del costo del cuaderno; los otros cinco modelos
quedan por debajo de 2 s en todos los brazos. En total, 14.6 min de ajustes y 20 min de reloj.

### Por origen (exploratorio, no decide)

El desglose repite un patrón que el 04c ya había mostrado, y es el que más conviene no perder de
vista: **donde las huellas ayudan de forma amplia no es donde el proyecto las necesita**. En los
compuestos sin equivalente natural mejoran el MCC de cinco de los seis modelos (svm_rbf
0.499 → 0.548, árbol 0.364 → 0.417, `bayes` 0.343 → 0.394, knn 0.476 → 0.510), y en los naturales
exactos y en los análogos estereoquímicos no mejoran nada.

El 04c muestra la misma forma, y conviene decirla con cuidado porque su propia lectura todavía no
está escrita. Contando sobre `notebooks/datos/04c_brazos.csv` de la corrida
`2026-10-28-artemio-1.3-comparativa-regresion-huellas`, en los pares que solo tiene ChEMBL las
huellas mejoran la concordancia de cinco de los seis modelos, entre +0.016 y +0.045; en los que
solo tiene NPASS mejoran dos de seis, y el techo del estrato sigue siendo el `mlp` con
descriptores solos. Así que no es que en NPASS no haya ganancia —el kNN sube +0.085 ahí, la mayor
de todo el 04c— sino que está en un modelo y no en la familia. La ganancia amplia y consistente
está del lado no natural en los dos cuadernos.

### Para el Avance 5

**No se agregan las huellas a los ensambles de la fase 1 por desempeño**, porque no lo mejoran y
sí aumentan el sobreajuste. El argumento que queda en pie es el de la explicabilidad: una columna
Sort & Slice es una subestructura concreta y permite una atribución a nivel de fragmento, que la
decisión 1.3 pide y los descriptores no dan. Si se usan para eso, se dice así —para explicar, no
porque midan mejor— y se reportan los dos brazos.

Queda abierto si merece la pena un brazo con huellas solo para el estrato sin equivalente natural.
No se decide aquí: el desglose es exploratorio y el umbral se fijó para el agregado.

# Declaración de uso de IA y contribución

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Nos apoyó con la generación de código de las huellas y de la depuración dentro del pliegue, y con el diseño de la comparación | Las secciones que nos ayudó a implementar fueron celdas de código | La decisión de medir las huellas, la regla de mejora, la revisión de los resultados y los comentarios generales fueron del equipo |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | *Por completar* |
| Artemio Santiago Padilla Robles | *Por completar* |
| José Antonio Vázquez Martínez | *Por completar* |